# Session 9 Practice — Measuring & Profiling Code Performance

> Companion to the note **Session-09-Implementation-and-Code-Sharing.md**.

**Goal of this notebook:** run every performance tool from the note on the *same* small ML task, so you can *see* what each one tells you and how they climb from coarse to fine:

```
time             → how long did the whole thing take?          (1 run, rough)
timeit           → how long on average, how consistent?         (many runs, avg ± std)
cProfile         → WHICH FUNCTION is the bottleneck?            (per-function time)
line_profiler    → WHICH LINE inside it is slow?                (per-line time)
memory_profiler  → which line eats MEMORY?                      (per-line memory)
```

**How to use this file:** each section has a **🧩 Your turn** cell (try it yourself first) followed by a **✅ Solution** cell (filled in, so you can review later). Try the first, then run/compare with the second.

> 📎 **Golden rule:** *measure first.* Never guess where the slow part is.

## 0 · Setup

We use scikit-learn's tiny **Iris** dataset (150 samples, 4 features, 3 classes) and a small **RandomForest** — exactly the examples from the note. Run this cell once.

In [ ]:
# If anything is missing, uncomment the installs (line_profiler & memory_profiler are the extras):
# %pip install scikit-learn line_profiler memory_profiler

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

print("Setup OK — sklearn imported.")

---
## 1 · `time` — how long did the whole thing take?

The simplest tool: record a timestamp before and after, then subtract. **Runs the code once**, so background noise can skew it.

**🧩 Your turn:** time how long it takes to `fit` a `DecisionTreeClassifier` on Iris. Fill in the `# TODO` lines.

In [ ]:
import time

X, y = load_iris(return_X_y=True)

start = None          # TODO: record the start time
model = DecisionTreeClassifier(random_state=42)
model.fit(X, y)       # the code being timed
end = None            # TODO: record the end time

# TODO: print the elapsed time (end - start) to 6 decimal places


In [ ]:
# ✅ Solution
import time

X, y = load_iris(return_X_y=True)

start = time.time()                       # record start
model = DecisionTreeClassifier(random_state=42)
model.fit(X, y)                           # the code being timed
end = time.time()                         # record end

print(f"Training Time: {end - start:.6f} seconds")
# One run only — run the cell twice and notice the number wobbles. That's why we need timeit next.

---
## 2 · `timeit` — how long on average, how consistent?

`timeit` runs the snippet **many times and averages**, so background noise cancels out. You get **average ± standard deviation**.

- `number=100` → each benchmark runs the function 100 times.
- `repeat=10` → run that whole benchmark 10 separate times.

**🧩 Your turn:** benchmark a `train_model()` function and print the mean + std dev per run.

In [ ]:
import timeit, statistics

def train_model():
    X, y = load_iris(return_X_y=True)
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X, y)

times = None          # TODO: use timeit.repeat(...) with repeat=10, number=100
avg_times = None      # TODO: convert each total to an average per single run (divide by 100)

# TODO: print the mean and the standard deviation of avg_times


In [ ]:
# ✅ Solution
import timeit, statistics

def train_model():
    X, y = load_iris(return_X_y=True)
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X, y)

# repeat=10 → run the whole benchmark 10 times; number=100 → each runs the fn 100 times
times = timeit.repeat(train_model, repeat=10, number=100)
avg_times = [t / 100 for t in times]      # total → average per single run

mean = statistics.mean(avg_times)
std = statistics.stdev(avg_times)
print(f"Average Training Time : {mean * 1000:.3f} ms")
print(f"Standard Deviation    : {std * 1000:.3f} ms")
# Small std dev → stable/consistent. Large std dev → noisy (CPU load, background processes, caching).

### 2b · The `%%timeit` cell magic (Jupyter/Colab)

Same idea, zero boilerplate: put `%%timeit` at the **top** of a cell and Jupyter benchmarks the whole cell for you. Run it and read the `mean ± std` line it prints.

In [ ]:
%%timeit
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
X, y = load_iris(return_X_y=True)
model = DecisionTreeClassifier(random_state=42)
model.fit(X, y)
# → e.g. 3.43 ms ± 160 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)

---
## 3 · `cProfile` — WHICH FUNCTION is the bottleneck?

`timeit` gives one number for the whole snippet. For a **full workflow** you want to know *which function* eats the time. `cProfile` is Python's built-in profiler: it tracks the **time + call count of every function**.

Column cheat-sheet:

| Column | Meaning |
|--------|---------|
| `ncalls` | how many times the function was called |
| `tottime` | time spent **only** in this function (excludes children) |
| `cumtime` | time in this function **+ everything it called** |

**🧩 Your turn:** profile a full `train_and_score()` workflow (load → split → fit → score) and sort by cumulative time.

In [ ]:
import cProfile, pstats

def train_and_score():
    X, y = load_iris(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
    model = RandomForestClassifier(n_estimators=5, max_depth=2, random_state=42)
    model.fit(X_train, y_train)
    return model.score(X_test, y_test)

# TODO: run train_and_score() under cProfile and print stats sorted by 'cumulative' time


In [ ]:
# ✅ Solution
import cProfile, pstats, io

def train_and_score():
    X, y = load_iris(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
    model = RandomForestClassifier(n_estimators=5, max_depth=2, random_state=42)
    model.fit(X_train, y_train)
    return model.score(X_test, y_test)

profiler = cProfile.Profile()
accuracy = profiler.runcall(train_and_score)   # profile just this call
print(f"Accuracy: {accuracy:.2f}\n")

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats("cumulative")
stats.print_stats(12)                          # top 12 rows by cumulative time
print(stream.getvalue())
# Read the top rows: fit (RandomForest) should consume the most cumulative time — that's the bottleneck function.

---
## 4 · `line_profiler` — WHICH LINE inside it is slow?

`cProfile` says *which function*. `line_profiler` zooms in to *which line inside that function* — far more readable, line-by-line.

In a notebook we load its magic with `%load_ext line_profiler`, then run `%lprun -f <function> <call>`.

**🧩 Your turn:** line-profile `train_and_score` and find the line with the highest `% Time`.

In [ ]:
# TODO: load the line_profiler extension, then run %lprun on train_and_score
# Hint:  %load_ext line_profiler
#        %lprun -f train_and_score train_and_score()


In [ ]:
# ✅ Solution
%load_ext line_profiler
%lprun -f train_and_score train_and_score()
# In the popup/output table, scan the '% Time' column.
# model.fit(...) should be the biggest share (~60%+) — that's the line to optimize.

---
## 5 · `memory_profiler` — which line eats MEMORY?

Time isn't the only resource. `memory_profiler` measures **memory per line**. (The note's `memray` is Mac/Linux-only; **`memory_profiler` is the Windows-friendly choice** — that's what we use here.)

Column cheat-sheet: **Mem usage** = memory used now · **Increment** = extra memory *this line* added · **Occurrences** = times the line ran.

**🧩 Your turn:** use the `%mprun` magic to see per-line memory of `train_and_score`.

In [ ]:
# TODO: load the memory_profiler extension, then run %mprun on train_and_score
# Hint:  %load_ext memory_profiler
#        %mprun -f train_and_score train_and_score()
# Note: %mprun needs the target function defined in an imported file; the solution cell handles that.


In [ ]:
# ✅ Solution (part A) — %mprun needs the function in a real module, so write one to disk first.
code = '''
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

def train_and_score():
    X, y = load_iris(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
    model = RandomForestClassifier(n_estimators=5, max_depth=2, random_state=42)
    model.fit(X_train, y_train)
    return model.score(X_test, y_test)
'''
with open("mem_demo.py", "w") as f:
    f.write(code)
print("wrote mem_demo.py")

In [ ]:
# ✅ Solution (part B)
%load_ext memory_profiler
from mem_demo import train_and_score as train_and_score_mem
%mprun -f train_and_score_mem train_and_score_mem()
# Scan the 'Increment' column: model.fit(...) should show the biggest memory bump.
#
# Alternative that works even outside Jupyter: decorate the function with @profile and run
#   python -m memory_profiler mem_demo.py
# from a terminal.

---
## 🎯 Recap — the performance-analysis ladder

You just climbed every rung on the *same* task:

| Rung | Tool | Question it answers |
|------|------|---------------------|
| 1 | `time` | How long did the whole thing take? (1 run, rough) |
| 2 | `timeit` / `%%timeit` | How long on average, how consistent? (avg ± std) |
| 3 | `cProfile` | **Which function** is the bottleneck? |
| 4 | `line_profiler` | **Which line** inside it is slow? |
| 5 | `memory_profiler` | Which line eats **memory**? |

**Takeaway:** start coarse, confirm with a benchmark, locate the slow *function*, zoom to the slow *line*, and check *memory* separately — always **measure before you optimize**.